# Lab 02_1. 선형회귀의 순전파와 손실

## 목표

- Linear Regression(리니어 리그레션, 선형회귀)의 역할 이해하기
- Hypothesis(하이파서시스, 가설) $H(x)=Wx+b$ 이해하기
- Prediction(프리딕션, 예측값)과 실제값의 차이 계산하기
- MSE(엠에스이, 평균제곱오차)로 Loss(로스, 손실) 계산하기
- 순전파 → 손실 → 역전파 → 매개변수 갱신 흐름 익히기
- 학습된 $W$와 $b$로 새로운 값 예측하기

> **A model learns by reducing the difference between prediction and target.**  
> **모델은 예측값과 실제값의 차이를 줄이는 방향으로 학습합니다.**


In [ ]:
import torch

# 재현성: 난수 고정
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())


## 실습 단계

### 1. 선형회귀 문제 이해하기

Linear Regression(리니어 리그레션, 선형회귀)은 입력 $x$로부터 연속적인 값 $y$를 예측합니다.

- Input(인풋, 입력) $x$: 예측에 사용하는 값
- Target(타깃, 실제값) $y$: 모델이 맞혀야 하는 정답
- Prediction(프리딕션, 예측값) $\hat{y}$: 모델이 계산한 값
- Regression(리그레션, 회귀): 온도·전압·속도·가격과 같은 연속적인 수치 예측

이번 실습은 원리를 분명하게 보기 위해 다음 세 점을 사용합니다.

- 입력 1 → 실제값 1
- 입력 2 → 실제값 2
- 입력 3 → 실제값 3


In [ ]:
# 학습 데이터
x_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
y_train = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)

print("x_train:", x_train)
print("y_train:", y_train)
print("shape:", x_train.shape, y_train.shape)


### 2. Hypothesis: 직선으로 관계 표현하기

선형회귀는 데이터의 관계를 다음 직선으로 가정합니다.

$$
H(x)=Wx+b
$$

- $H(x)$ 또는 $\hat{y}$: 가설의 출력인 예측값
- $W$: Weight(웨이트, 가중치), 직선의 기울기
- $b$: Bias(바이어스, 편향), 직선의 $y$절편
- $x$: 모델에 입력하는 값

$W$가 바뀌면 직선의 기울기가 바뀌고, $b$가 바뀌면 직선이 위아래로 이동합니다.

> **Forward computes predictions from inputs and parameters.**  
> **Forward(포워드, 순전파)는 입력과 매개변수로부터 예측값을 계산합니다.**


In [ ]:
# 순전파 직접 계산
W_example = torch.tensor(0.5)
b_example = torch.tensor(2.0)

y_pred_example = W_example * x_train + b_example

print("가설: H(x) = 0.5x + 2.0")
print("예측값:", y_pred_example)
print("실제값:", y_train)
print("오차:", y_pred_example - y_train)


### 3. Loss: 예측이 얼마나 틀렸는지 계산하기

MSE(엠에스이, 평균제곱오차)는 각 예측 오차를 제곱한 뒤 평균을 계산합니다.

$$
\mathrm{MSE}=
\frac{1}{m}\sum_{i=1}^{m}
\left(H\left(x^{(i)}\right)-y^{(i)}\right)^2
$$

1. $H(x)-y$: 예측값과 실제값의 차이 계산
2. $(H(x)-y)^2$: 양수·음수 오차의 상쇄 방지
3. 평균: 데이터 한 개당 제곱 오차 계산

Loss가 작을수록 현재 직선이 데이터에 더 잘 맞습니다.


In [ ]:
# MSE 단계별 계산
errors = y_pred_example - y_train
squared_errors = errors ** 2
mse_manual = squared_errors.mean()

print("오차:", errors)
print("제곱 오차:", squared_errors)
print("MSE:", mse_manual.item())


### 4. 두 Hypothesis의 Loss 비교하기

사람은 그래프를 보고 좋은 직선을 고를 수 있지만, 컴퓨터는 Loss를 숫자로 비교합니다.


In [ ]:
def mse_for_line(x, y, weight, bias):
    # 예측·MSE 계산
    prediction = weight * x + bias
    return ((prediction - y) ** 2).mean()


loss_h1 = mse_for_line(x_train, y_train, weight=1.0, bias=0.0)
loss_h2 = mse_for_line(x_train, y_train, weight=0.5, bias=2.0)

print("H1(x) = 1.0x + 0.0 → MSE:", loss_h1.item())
print("H2(x) = 0.5x + 2.0 → MSE:", round(loss_h2.item(), 4))
print("더 좋은 가설:", "H1" if loss_h1 < loss_h2 else "H2")


### 5. 학습할 W와 b 준비하기

PyTorch에서 `requires_grad=True`를 지정하면 해당 Tensor의 Gradient(그레이디언트, 기울기)를 자동으로 계산할 수 있습니다.

초기값은 정답이 아닙니다. 학습을 반복하면서 Loss가 줄어드는 방향으로 $W$와 $b$가 바뀝니다.


In [ ]:
# 학습 매개변수: W·b
W = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

print("초기 W:", W.item())
print("초기 b:", b.item())
print("W의 자동미분 사용 여부:", W.requires_grad)


### 6. Forward → Loss → Backward 한 번 실행하기

학습의 기본 흐름은 다음 네 단계입니다.

1. `y_pred = W * x + b`: 순전파로 예측값 계산
2. `loss = ((y_pred - y) ** 2).mean()`: 손실 계산
3. `loss.backward()`: 기울기 계산
4. `optimizer.step()`: $W$와 $b$ 갱신

`backward()`는 Loss를 줄이려면 $W$와 $b$를 어느 방향으로 바꿔야 하는지 계산합니다.


In [ ]:
# 1. 순전파
y_pred = W * x_train + b

# 2. 손실 계산
loss = ((y_pred - y_train) ** 2).mean()

# 3. 역전파
loss.backward()

print("예측값:", y_pred.detach())
print("Loss:", loss.item())
print("W.grad:", W.grad.item())
print("b.grad:", b.grad.item())


### 7. Optimizer로 W와 b 학습하기

SGD(에스지디, 확률적 경사하강법)는 기울기의 반대 방향으로 매개변수를 조금씩 이동합니다.

$$
\text{새 매개변수}
\leftarrow
\text{현재 매개변수}
-
\text{learning rate}\times\text{gradient}
$$

Learning Rate(러닝 레이트, 학습률)는 한 번에 이동하는 크기입니다. 이번 실습에서는 `0.01`을 사용합니다.

> **Training repeats forward, loss, backward, and update.**  
> **학습은 Forward → Loss → Backward → Update를 반복합니다.**


In [ ]:
# W·b 초기화
W = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
b = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# 최적화 도구
optimizer = torch.optim.SGD([W, b], lr=0.01)

loss_history = []

for step in range(2001):
    # 기울기 초기화
    optimizer.zero_grad()

    # 순전파
    y_pred = W * x_train + b

    # 손실 계산
    loss = ((y_pred - y_train) ** 2).mean()

    # 역전파
    loss.backward()

    # 매개변수 갱신
    optimizer.step()

    loss_history.append(loss.item())

    if step % 400 == 0:
        print(
            f"step {step:4d} | "
            f"loss {loss.item():.8f} | "
            f"W {W.item():.4f} | b {b.item():.4f}"
        )


### 8. 학습 결과 해석하기

데이터가 $(1,1)$, $(2,2)$, $(3,3)$이므로 가장 잘 맞는 직선은 $H(x)=1x+0$입니다.

학습이 정상적으로 진행되면 다음 변화가 나타납니다.

- Loss는 0에 가까워집니다.
- $W$는 1에 가까워집니다.
- $b$는 0에 가까워집니다.


In [ ]:
print("학습된 W:", W.item())
print("학습된 b:", b.item())
print("최종 Loss:", loss_history[-1])
print("Loss 감소:", loss_history[0], "→", loss_history[-1])


### 9. 새로운 입력값 예측하기

학습할 때 사용하지 않은 입력값도 학습된 직선에 넣어 예측할 수 있습니다.

예측 단계에서는 기울기 계산이 필요하지 않으므로 `torch.no_grad()`를 사용합니다.


In [ ]:
x_new = torch.tensor([5.0, 2.5, 1.5, 3.5], dtype=torch.float32)

with torch.no_grad():
    y_new = W * x_new + b

print("새 입력:", x_new)
print("예측 결과:", y_new)


### 10. 슬라이드의 새 학습 데이터 적용하기

슬라이드의 두 번째 예제는 $y=x+1.1$에 가까운 데이터를 사용합니다.

- 입력 1 → 실제값 2.1
- 입력 2 → 실제값 3.1
- 입력 3 → 실제값 4.1
- 입력 4 → 실제값 5.1
- 입력 5 → 실제값 6.1

반복 학습 코드를 함수로 묶으면 데이터가 바뀌어도 같은 절차를 재사용할 수 있습니다.


In [ ]:
def train_linear_regression(x, y, learning_rate=0.01, epochs=3000):
    # W·b 초기화
    weight = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
    bias = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
    optimizer = torch.optim.SGD([weight, bias], lr=learning_rate)

    for _ in range(epochs):
        optimizer.zero_grad()
        prediction = weight * x + bias
        loss = ((prediction - y) ** 2).mean()
        loss.backward()
        optimizer.step()

    return weight.detach(), bias.detach(), loss.detach()


x_train_2 = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)
y_train_2 = torch.tensor([2.1, 3.1, 4.1, 5.1, 6.1], dtype=torch.float32)

W_2, b_2, loss_2 = train_linear_regression(x_train_2, y_train_2)

with torch.no_grad():
    prediction_at_7 = W_2 * torch.tensor(7.0) + b_2

print("학습된 W:", W_2.item())
print("학습된 b:", b_2.item())
print("최종 Loss:", loss_2.item())
print("x=7 예측:", prediction_at_7.item())


## 확인 실습

센서의 입력 전압과 실제 측정값 사이의 관계를 선형회귀로 보정합니다.

- 입력 전압 0.5 V → 실제 측정값 11
- 입력 전압 1.0 V → 실제 측정값 21
- 입력 전압 1.5 V → 실제 측정값 31
- 입력 전압 2.0 V → 실제 측정값 41

아래 코드의 `TODO`를 확인하고 실행합니다.

- 학습 데이터 Tensor 만들기
- 제공된 함수로 $W$와 $b$ 학습하기
- 입력 전압 1.25 V에 대한 측정값 예측하기


In [ ]:
# TODO 1: 학습 데이터
voltage_data = torch.tensor([0.5, 1.0, 1.5, 2.0], dtype=torch.float32)
measured_data = torch.tensor([11.0, 21.0, 31.0, 41.0], dtype=torch.float32)

# TODO 2: 모델 학습
sensor_W, sensor_b, sensor_loss = train_linear_regression(
    voltage_data,
    measured_data,
    learning_rate=0.05,
    epochs=4000,
)

# TODO 3: 1.25 V 예측
new_voltage = torch.tensor(1.25, dtype=torch.float32)
with torch.no_grad():
    corrected_value = sensor_W * new_voltage + sensor_b

print("sensor W:", sensor_W.item())
print("sensor b:", sensor_b.item())
print("sensor Loss:", sensor_loss.item())
print("1.25 V 예측값:", corrected_value.item())


## 실행 확인

아래 셀에서 오류가 없으면 `Lab 02_1`이 정상적으로 완료된 것입니다.


In [ ]:
assert x_train.shape == torch.Size([3])
assert y_pred_example.shape == torch.Size([3])
assert abs(mse_manual.item() - (3.5 / 3.0)) < 1e-6
assert loss_h1.item() < loss_h2.item()
assert loss_history[-1] < loss_history[0]
assert abs(W.item() - 1.0) < 0.02
assert abs(b.item()) < 0.05
assert abs(W_2.item() - 1.0) < 0.02
assert abs(b_2.item() - 1.1) < 0.05
assert abs(corrected_value.item() - 26.0) < 0.2

print("Lab 02_1 실행 확인 완료")


## 핵심 정리

- 데이터: 입력 $x$와 실제값 $y$를 Tensor로 준비
- 가설: $H(x)=Wx+b$로 예측값 계산
- 손실: MSE로 예측값과 실제값의 차이 계산
- 역전파: 자동미분으로 $W$와 $b$의 기울기 계산
- 갱신: SGD로 $W$와 $b$ 갱신
- 반복 학습: 손실이 감소하도록 학습 과정 반복
- 예측: 학습된 $W$와 $b$로 새로운 입력 계산

> **Linear Regression = Forward → Loss → Backward → Update**

## 다음 단계

다음 주제는 새 노트북 `Lab_02_2_Linear_Regression_Neural_Network.ipynb`로 진행합니다.

- $W$ 변화에 따른 Cost 곡선 확인
- Gradient(그레이디언트, 기울기)의 방향 이해
- Gradient Descent(그레이디언트 디센트, 경사하강법) 직접 구현
- Learning Rate(러닝 레이트, 학습률)가 너무 작거나 클 때의 차이 비교
